In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.silver")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
w_seq  = Window.partitionBy("currency").orderBy("date")
w_5d   = Window.partitionBy("currency").orderBy("date").rowsBetween(-4, 0)
w_20d  = Window.partitionBy("currency").orderBy("date").rowsBetween(-19, 0)

df = (
    spark.table("stocks.bronze.fx_exchange_rates")
    .withColumn("prev_rate",      F.lag("rate", 1).over(w_seq))
    .withColumn("daily_change",   (F.col("rate") - F.col("prev_rate")) / F.col("prev_rate"))
    .withColumn("rate_5d_avg",    F.avg("rate").over(w_5d))
    .withColumn("rate_20d_avg",   F.avg("rate").over(w_20d))
    .drop("prev_rate", "ingested_at")
)

In [ ]:
(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("stocks.silver.fx_exchange_rates")
)

print(f"stocks.silver.fx_exchange_rates: {spark.table('stocks.silver.fx_exchange_rates').count()} rows")
spark.table("stocks.silver.fx_exchange_rates").orderBy("date", "currency").show(10)